## SQL con Spark

Una de las grandes ventajas de Spark es que no necesitamos aprender una API nueva para hacer consultas complejas. Spark nos permite ejecutar **SQL** estándar directamente sobre nuestros _DataFrames_, lo que facilita mucho la vida a quienes ya están familiarizados con bases de datos relacionales.

En este artículo vamos a ver cómo combinar los datos de los taxis verdes y amarillos de Nueva York, registrarlos como una vista temporal y consultarlos con SQL para calcular métricas de ingresos mensuales por zona.

### Crear la sesión de Spark

Lo primero es siempre crear la sesión de Spark. Aquí usamos la variable de entorno `SPARK_MASTER` para apuntar al nodo maestro del clúster. El nombre de la aplicación (`appName`) es el que veremos en la interfaz gráfica de Spark.

In [1]:
import pyspark
import os
from pyspark.sql import SparkSession

# Creamos una sesión PySpark contra nuestro servicio "spark-master"
spark = SparkSession.builder \
    .master(os.environ.get('SPARK_MASTER')) \
    .appName('sql-en-spark') \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/06 19:26:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Leer los datos en formato Parquet

Leemos todos los archivos Parquet de los taxis verdes y amarillos. El patrón `/*/*` nos permite recorrer los subdirectorios de año y mes sin tener que especificar cada ruta manualmente.

In [4]:
df_green = spark.read.parquet('/data/parquet/green/*/*')
df_yellow = spark.read.parquet('/data/parquet/yellow/*/*')

### Inspeccionar los esquemas

Antes de combinar dos _DataFrames_, conviene revisar sus esquemas. Aquí notamos que las columnas de fecha y hora tienen nombres distintos: los taxis verdes usan `lpep_pickup_datetime` y `lpep_dropoff_datetime`, mientras que los amarillos usan `tpep_pickup_datetime` y `tpep_dropoff_datetime`. Para poder combinar ambos conjuntos de datos, necesitaremos unificar esos nombres.

In [5]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [6]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



### Encontrar las columnas comunes

Usamos la intersección de conjuntos de Python para encontrar las columnas que comparten ambos _DataFrames_. Esto nos da la lista de campos que podemos seleccionar de los dos sin preocuparnos por diferencias de esquema.

En este punto, las columnas de fecha y hora todavía no aparecen en la intersección porque tienen nombres distintos en cada _DaraFrame_.

In [7]:
set(df_green.columns) & set(df_yellow.columns)

{'DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'trip_distance'}

### Renombrar las columnas de fecha y hora

Con `withColumnRenamed` le damos a las columnas de fecha un nombre común en ambos _DataFrames_. Si volvemos a calcular la intersección de columnas tras este cambio, veremos que `pickup_datetime` y `dropoff_datetime` ya aparecen.

In [15]:
df_green = df_green.withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime')
df_green = df_green.withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

In [14]:
df_yellow = df_yellow.withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime')
df_yellow = df_yellow.withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

In [16]:
set(df_green.columns) & set(df_yellow.columns)

{'DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'dropoff_datetime',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'pickup_datetime',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'trip_distance'}

### Definir las columnas a seleccionar

Definimos la lista de columnas que nos interesa. Esta lista solo incluye las columnas comunes, descartando las que son exclusivas de cada tipo de taxi (como `trip_type` o `ehail_fee`, que solo existen en los taxis verdes).

In [20]:
sorted_columns = [
    'VendorID',
    'pickup_datetime',
    'dropoff_datetime',
    'store_and_fwd_flag',
    'RatecodeID',
    'PULocationID',
    'DOLocationID',
    'passenger_count',
    'trip_distance',
    'fare_amount',
    'extra',
    'mta_tax',
    'tip_amount',
    'tolls_amount',
    'improvement_surcharge',
    'total_amount',
    'payment_type',
    'congestion_surcharge',
]

### Combinar los dos DataFrames en uno

Aquí hacemos varias cosas a la vez:

1. Seleccionamos solo las columnas que nos interesan con `select(sorted_columns)`.
2. Añadimos una columna `service_type` con `F.lit(...)` para identificar el origen de cada registro. `lit` crea una columna con un valor constante (una cadena de texto literal).
3. Combinamos los dos _DataFrames_ con `unionAll`, que apila las filas de ambos en uno solo.
4. Verificamos el resultado con un `groupBy` y `count` para confirmar cuántos registros hay de cada tipo.

In [24]:
from pyspark.sql import functions as F

df_green_select = df_green \
    .select(sorted_columns) \
    .withColumn('service_type', F.lit('green'))

df_yellow_select = df_yellow \
    .select(sorted_columns) \
    .withColumn('service_type', F.lit('yellow'))

df_trips = df_green_select.unionAll(df_yellow_select)

df_trips.groupBy('service_type').count().show()

+------------+---------+
|service_type|    count|
+------------+---------+
|       green|  8348567|
|      yellow|124048218|
+------------+---------+



### Registrar el DataFrame como vista temporal

Esta es la clave para poder usar SQL en Spark. `createOrReplaceTempView` registra el _DataFrame_ como una vista temporal con el nombre que le indiquemos. A partir de aquí podemos referirnos a él como si fuera una tabla en cualquier consulta SQL.

> La vista es temporal. Solo existe mientras dure la sesión de Spark.

In [26]:
df_trips.createOrReplaceTempView('trips')

### Consultar la vista con SQL

Con `spark.sql(...)` ejecutamos cualquier consulta SQL estándar sobre las vistas registradas. El resultado es un _DataFrame_ normal de Spark, así que podemos encadenarle cualquier operación adicional.

In [43]:
df_trips.count()

132396785

In [42]:
spark.sql('SELECT PULocationID, DOLocationID, pickup_datetime, service_type FROM trips').head(5)

[Row(PULocationID=82, DOLocationID=160, pickup_datetime=datetime.datetime(2019, 1, 27, 18, 7, 53), service_type='green'),
 Row(PULocationID=41, DOLocationID=74, pickup_datetime=datetime.datetime(2019, 1, 17, 7, 38, 57), service_type='green'),
 Row(PULocationID=82, DOLocationID=129, pickup_datetime=datetime.datetime(2019, 1, 30, 0, 40, 5), service_type='green'),
 Row(PULocationID=130, DOLocationID=10, pickup_datetime=datetime.datetime(2019, 1, 29, 18, 59, 48), service_type='green'),
 Row(PULocationID=119, DOLocationID=226, pickup_datetime=datetime.datetime(2019, 1, 11, 11, 56, 18), service_type='green')]

### Definir una consulta de agregación

Aquí vemos una de las grandes ventajas de usar SQL en Spark: podemos escribir consultas complejas con una sintaxis familiar. Esta consulta agrupa los viajes por zona de recogida, mes y tipo de servicio, y calcula la suma de cada componente del importe total más algunas medias.

`DATE_TRUNC('month', pickup_datetime)` trunca la fecha al primer día del mes, lo que nos permite agrupar todos los viajes de un mismo mes independientemente del día exacto.

In [29]:
grouped_trips = """
SELECT 
    -- Criterios de agrupación
    PULocationID AS revenue_zone,
    DATE_TRUNC('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Ingresos
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Otras métricas
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips
GROUP BY
    revenue_zone, revenue_month, service_type
"""

### Ejecutar la consulta

Pasamos la cadena SQL a `spark.sql(...)` y obtenemos un _DataFrame_ con los resultados de la agregación. Al igual que cualquier transformación en Spark, esto se evalúa de forma perezosa: el trabajo real no ocurre hasta que ejecutemos una acción sobre `df_grouped`.

In [34]:
df_grouped = spark.sql(grouped_trips)

### Escribir el resultado en Parquet

Guardamos el resultado final en un único archivo Parquet. `coalesce(1)` reduce el número de particiones a una sola antes de escribir, lo que es útil cuando el resultado es lo suficientemente pequeño para caber en un archivo y queremos evitar que Spark lo divida en múltiples ficheros. El parámetro `mode='overwrite'` permite sobreescribir la carpeta de destino si ya existe.

In [37]:
df_grouped.coalesce(1).write.parquet('/data/report/revenue', mode='overwrite')